In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import pickle

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'H:\DPJI\IDDPedestrian\annotations\gopro',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'obs_len': 15, 
    'pred_len': 45,
    'seq_len': 60,
    'hidden_size': 256,   # Increased for better capacity
    'latent_dim': 64,     # Increased latent space
    'batch_size': 128,    # Larger batch size helps CVAEs
    'epochs': 60,         # Needs more epochs to converge on velocities
    'lr': 1e-3,
    'best_k': 20,         # For evaluation
    'vel_scale': 20.0,    # Scaling factor for velocities to help convergence
    'kl_weight': 1.0      # Max KL weight
}
print(f"✅ BiTraP Config Loaded. Device: {CONFIG['device']}")

✅ BiTraP Config Loaded. Device: cuda


In [2]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, vel_scale=1.0):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.vel_scale = vel_scale
        self.samples = []
        
        print("📂 Parsing XMLs for BiTraP Trajectories...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                for track in root.findall('track'):
                    if track.attrib['label'] != 'pedestrian': continue
                    track_data = []
                    # 1920x1080 resolution assumption based on IDD
                    W, H = 1920.0, 1080.0
                    
                    for box in track.findall('box'):
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        # Center Bottom (Feet)
                        cx = (xtl + xbr) / 2.0
                        cy = ybr 
                        w = xbr - xtl
                        h = ybr - ytl
                        track_data.append([cx/W, cy/H, w/W, h/H]) # Norm Coords
                    
                    track_data = np.array(track_data)
                    if len(track_data) < self.seq_len: continue
                    
                    # Sliding window
                    for i in range(0, len(track_data) - self.seq_len + 1, 15):
                        seq = track_data[i : i+self.seq_len]
                        
                        # Absolute positions (for GT/Vis)
                        obs_abs = seq[:obs_len]
                        pred_abs = seq[obs_len:]
                        
                        # Calculate Velocities (Offsets)
                        # v_t = p_t - p_{t-1}
                        # We pad the first frame velocity with 0
                        vel_seq = np.zeros_like(seq)
                        vel_seq[1:] = seq[1:] - seq[:-1]
                        vel_seq[0] = vel_seq[1] # Approx first frame
                        
                        # Scale Velocities
                        vel_seq = vel_seq * self.vel_scale
                        
                        self.samples.append({
                            'obs_vel': vel_seq[:obs_len, 0:2], # Only X,Y vel
                            'pred_vel': vel_seq[obs_len:, 0:2],
                            'obs_abs': obs_abs,                # Keep abs for reconstruction
                            'pred_abs': pred_abs
                        })
            except Exception as e: pass
                
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            torch.tensor(item['obs_vel'], dtype=torch.float32),
            torch.tensor(item['pred_vel'], dtype=torch.float32),
            torch.tensor(item['obs_abs'], dtype=torch.float32),
            torch.tensor(item['pred_abs'], dtype=torch.float32)
        )

dataset = IDDTrajectoryDataset(CONFIG['xml_root'], CONFIG['obs_len'], CONFIG['pred_len'], CONFIG['vel_scale'])
print(f"Dataset Size: {len(dataset)}")

📂 Parsing XMLs for BiTraP Trajectories...


  0%|          | 0/33 [00:00<?, ?it/s]

Dataset Size: 18587


In [3]:
class BiTraP_CVAE(nn.Module):
    def __init__(self, input_dim=2, hidden_size=256, latent_dim=64):
        super(BiTraP_CVAE, self).__init__()
        
        # 1. Past Encoder (Condition)
        self.past_encoder = nn.LSTM(input_dim, hidden_size, batch_first=True)
        
        # 2. Future Encoder (Training Only - for Posterior)
        self.future_encoder = nn.LSTM(input_dim, hidden_size, batch_first=True)
        
        # 3. Latent Space
        self.fc_mu = nn.Linear(hidden_size * 2, latent_dim) 
        self.fc_logvar = nn.Linear(hidden_size * 2, latent_dim)
        
        # Prior Network (P(z|X))
        self.fc_mu_prior = nn.Linear(hidden_size, latent_dim)
        self.fc_logvar_prior = nn.Linear(hidden_size, latent_dim)
        
        # 4. Decoder
        self.decoder_fc = nn.Linear(latent_dim + hidden_size, hidden_size)
        self.decoder = nn.LSTM(input_dim, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, input_dim)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, obs_vel, target_vel=None, training=True):
        # obs_vel: [B, 15, 2]
        _, (h_past, _) = self.past_encoder(obs_vel)
        h_past = h_past[-1] # [B, Hidden]
        
        if training and target_vel is not None:
            # Posterior Q(z|X,Y)
            _, (h_fut, _) = self.future_encoder(target_vel)
            h_fut = h_fut[-1]
            h_combined = torch.cat([h_past, h_fut], dim=1)
            mu = self.fc_mu(h_combined)
            logvar = self.fc_logvar(h_combined)
        else:
            # Prior P(z|X)
            mu = self.fc_mu_prior(h_past)
            logvar = self.fc_logvar_prior(h_past)
            
        z = self.reparameterize(mu, logvar)
        
        # Init Decoder
        # Combine Z with Past context
        decoder_init = torch.tanh(self.decoder_fc(torch.cat([z, h_past], dim=1)))
        
        # Prepare hidden states for LSTM [1, B, H]
        h_state = decoder_init.unsqueeze(0)
        c_state = torch.zeros_like(h_state)
        
        outputs = []
        # First input to decoder is the last observed velocity
        curr_input = obs_vel[:, -1, :].unsqueeze(1) 
        
        pred_len = target_vel.size(1) if target_vel is not None else CONFIG['pred_len']
        
        for _ in range(pred_len):
            out, (h_state, c_state) = self.decoder(curr_input, (h_state, c_state))
            pred_vel = self.fc_out(out) # Predict offset
            outputs.append(pred_vel)
            curr_input = pred_vel # Autoregressive
            
        return torch.cat(outputs, dim=1), mu, logvar

model_bitrap = BiTraP_CVAE(
    hidden_size=CONFIG['hidden_size'],
    latent_dim=CONFIG['latent_dim']
).to(CONFIG['device'])

print("✅ BiTraP Model Initialized (Velocity-based).")

✅ BiTraP Model Initialized (Velocity-based).


In [4]:
def elbo_loss(recon_vel, target_vel, mu, logvar, kl_weight=1.0):
    # MSE on Velocities
    MSE = nn.functional.mse_loss(recon_vel, target_vel, reduction='sum')
    # KL Divergence
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + (kl_weight * KLD), MSE.item(), KLD.item()

# Split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

optimizer = optim.Adam(model_bitrap.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

print("🚀 Starting BiTraP Training...")

for epoch in range(CONFIG['epochs']):
    model_bitrap.train()
    total_loss = 0
    total_mse = 0
    total_kld = 0
    
    # KL Annealing: Ramp up from 0.0 to 1.0 over first 10 epochs
    kl_w = min(1.0, (epoch + 1) / 10.0) * CONFIG['kl_weight']
    
    for obs_vel, target_vel, _, _ in tqdm(train_loader, leave=False):
        obs_vel = obs_vel.to(CONFIG['device'])
        target_vel = target_vel.to(CONFIG['device'])
        
        optimizer.zero_grad()
        preds, mu, logvar = model_bitrap(obs_vel, target_vel=target_vel, training=True)
        
        loss, mse_val, kld_val = elbo_loss(preds, target_vel, mu, logvar, kl_weight=kl_w)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model_bitrap.parameters(), 1.0)
        
        optimizer.step()
        total_loss += loss.item()
        total_mse += mse_val
        total_kld += kld_val
        
    # Validation
    model_bitrap.eval()
    val_loss = 0
    with torch.no_grad():
        for obs_vel, target_vel, _, _ in val_loader:
            obs_vel = obs_vel.to(CONFIG['device'])
            target_vel = target_vel.to(CONFIG['device'])
            preds, _, _ = model_bitrap(obs_vel, training=False)
            val_loss += nn.functional.mse_loss(preds, target_vel, reduction='sum').item()
            
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.1f} (MSE:{total_mse/len(train_loader):.1f} KL:{total_kld/len(train_loader):.1f}) | Val MSE: {avg_val_loss:.2f} | LR: {optimizer.param_groups[0]['lr']:.5f}")

torch.save(model_bitrap.state_dict(), "bitrap_model.pth")
print("💾 BiTraP Model Saved.")

🚀 Starting BiTraP Training...


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 1 | Loss: 36.4 (MSE:36.4 KL:0.3) | Val MSE: 29.85 | LR: 0.00100


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 2 | Loss: 27.2 (MSE:27.2 KL:0.1) | Val MSE: 22.95 | LR: 0.00100


...

Epoch 58 | Loss: 20.1 (MSE:20.1 KL:0.0) | Val MSE: 18.96 | LR: 0.00013


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 59 | Loss: 20.1 (MSE:20.1 KL:0.0) | Val MSE: 19.09 | LR: 0.00013


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 60 | Loss: 20.1 (MSE:20.1 KL:0.0) | Val MSE: 19.05 | LR: 0.00013
💾 BiTraP Model Saved.


In [5]:
def calculate_metrics_bitrap(model, loader):
    model.eval()
    mse_traj_list = [] 
    cmse_final_list = [] 
    cfmse_final_list = [] 
    
    W, H = 1920.0, 1080.0
    K = CONFIG['best_k']
    scale = CONFIG['vel_scale']
    
    with torch.no_grad():
        for obs_vel, target_vel, obs_abs, target_abs in tqdm(loader, desc="Evaluating"):
            obs_vel = obs_vel.to(CONFIG['device'])
            
            # Ground Truth Absolute Positions (Un-normalized to Pixels)
            gt_px = target_abs[:, :, 0:2].numpy() * [W, H]
            gt_wh_px = target_abs[:, :, 2:4].numpy() * [W, H]
            
            # Get last observed position to start integration [B, 2]
            last_obs_pos = obs_abs[:, -1, 0:2].to(CONFIG['device'])
            
            # Storage for Best-of-K
            best_mse_batch = np.full(obs_vel.size(0), float('inf'))
            best_pred_px = np.zeros((obs_vel.size(0), CONFIG['pred_len'], 2))
            
            for _ in range(K):
                # Predict Offsets
                pred_vel, _, _ = model(obs_vel, training=False)
                
                # 1. Unscale Velocity
                pred_vel = pred_vel / scale
                
                # 2. Integrate to get Absolute Positions
                # Cumulative Sum along time axis
                pred_cumsum = torch.cumsum(pred_vel, dim=1) 
                
                # Add to last observed position
                # last_obs_pos unsqueezed to [B, 1, 2] for broadcasting
                pred_pos = last_obs_pos.unsqueeze(1) + pred_cumsum
                
                # 3. Convert to Pixels [B, T, 2]
                pred_px_sample = pred_pos.cpu().numpy() * [W, H]
                
                # Calculate MSE (Avg Euclidean Distance Squared over time)
                # (x-x')^2 + (y-y')^2
                sq_diff = (pred_px_sample - gt_px)**2
                mse_per_traj = np.mean(np.sum(sq_diff, axis=2), axis=1) # [B]
                
                # Update Best
                improved = mse_per_traj < best_mse_batch
                best_mse_batch[improved] = mse_per_traj[improved]
                best_pred_px[improved] = pred_px_sample[improved]

            # --- Metrics Collection ---
            mse_traj_list.extend(best_mse_batch)
            
            # C-MSE (Final point error squared)
            pred_end = best_pred_px[:, -1, :]
            gt_end = gt_px[:, -1, :]
            c_mse = np.sum((pred_end - gt_end)**2, axis=1)
            cmse_final_list.extend(c_mse)
            
            # CF-MSE (Center + Foot)
            # Annotations: (cx, y_bottom). 
            # Foot is already (cx, y_bottom). Center is (cx, y_bottom - h/2)
            # Since X error is same for both, and Y error is same (offset by constant h/2),
            # Ideally CenterError == FootError.
            # However, strictly following formula:
            
            h_vec = gt_wh_px[:, -1, 1]
            
            # Pred Center
            pred_cy = pred_end[:, 1] - (h_vec / 2.0)
            pred_center = np.stack([pred_end[:, 0], pred_cy], axis=1)
            
            # GT Center
            gt_cy = gt_end[:, 1] - (h_vec / 2.0)
            gt_center = np.stack([gt_end[:, 0], gt_cy], axis=1)
            
            center_mse = np.sum((pred_center - gt_center)**2, axis=1)
            
            # CF-MSE = Center_MSE + Foot_MSE (which is C-MSE)
            cfmse_final_list.extend(center_mse + c_mse)

    return np.mean(mse_traj_list), np.mean(cmse_final_list), np.mean(cfmse_final_list)

# Run
mse, c_mse, cf_mse = calculate_metrics_bitrap(model_bitrap, val_loader)

print("="*40)
print(f"✅ FINAL RESULTS (Best-of-{CONFIG['best_k']}):")
print(f"   MSE (Avg):     {mse:.2f}")
print(f"   C-MSE (1.5s):  {c_mse:.2f}")
print(f"   CF-MSE (1.5s): {cf_mse:.2f}")
print("="*40)

with open('result_bitrap.pkl', 'wb') as f:
    pickle.dump({'MSE': mse, 'C-MSE': c_mse, 'CF-MSE': cf_mse}, f)

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

✅ FINAL RESULTS (Best-of-20):
   MSE (Avg):     3220.46
   C-MSE (1.5s):  13646.65
   CF-MSE (1.5s): 27293.29
